### Step -1 Define Schema as STRING

In [0]:

from pyspark.sql.types import StructType, StructField, StringType
customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_unique_id", StringType(), True),
    StructField("customer_zip_code_prefix", StringType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True)
])
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", StringType(), True),
    StructField("order_approved_at", StringType(), True),
    StructField("order_delivered_carrier_date", StringType(), True),
    StructField("order_delivered_customer_date", StringType(), True),
    StructField("order_estimated_delivery_date", StringType(), True)
])
order_items_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_item_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", StringType(), True),
    StructField("price", StringType(), True),
    StructField("freight_value", StringType(), True)
])
orders_payments_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("payment_sequential", StringType(), True),
    StructField("payment_type", StringType(), True),
    StructField("payment_installments", StringType(), True),
    StructField("payment_value", StringType(), True)
])
products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_category_name", StringType(), True),
    StructField("product_name_lenght", StringType(), True),
    StructField("product_description_lenght", StringType(), True),
    StructField("product_photos_qty", StringType(), True),
    StructField("product_weight_g", StringType(), True),
    StructField("product_length_cm", StringType(), True),
    StructField("product_height_cm", StringType(), True),
    StructField("product_width_cm", StringType(), True)
])

### Step-2 Reading CSV Files

In [0]:
customers_df= spark.read.format('csv').option("header", "true").load("/Volumes/ecommerce_pipeline/raw_data/raw_files/olist_customers_dataset.csv")
customers_df.createOrReplaceTempView("customers")
display(customers_df)


In [0]:
orders_df= spark.read.format('csv').option("header", "true").load("/Volumes/ecommerce_pipeline/raw_data/raw_files/olist_orders_dataset.csv")
orders_df.createOrReplaceTempView("orders")
display(orders_df)

In [0]:
orders_items_df= spark.read.format('csv').option("header", "true").load("/Volumes/ecommerce_pipeline/raw_data/raw_files/olist_order_items_dataset.csv")
orders_items_df.createOrReplaceTempView("orders")
display(orders_items_df)

In [0]:
orders_payments_df= spark.read.format('csv').option("header", "true").load("/Volumes/ecommerce_pipeline/raw_data/raw_files/olist_order_payments_dataset.csv")
orders_payments_df.createOrReplaceTempView("orders")
display(orders_payments_df)

In [0]:
products_df= spark.read.format('csv').option("header", "true").load("/Volumes/ecommerce_pipeline/raw_data/raw_files/olist_products_dataset.csv")
products_df.createOrReplaceTempView("orders")
display(products_df)

### Step - 3 Add Ingestion metadata

In [0]:
from pyspark.sql.functions import current_timestamp
customers_df = customers_df.withColumn('ingest_timestamp',current_timestamp())
orders_df = orders_df.withColumn('ingest_timestamp', current_timestamp())
orders_items_df = orders_items_df.withColumn('ingest_timestamp', current_timestamp())
orders_payments_df = orders_payments_df.withColumn('ingest_timestamp', current_timestamp())
products_df = products_df.withColumn('ingest_timestamp', current_timestamp())

### Step -4 Save AS Delta Table in Bronze

In [0]:
orders_df.write.format("delta").mode("overwrite").saveAsTable('ecommerce_pipeline.bronze.bronze_orders')
customers_df.write.format("delta").mode("overwrite").saveAsTable('ecommerce_pipeline.bronze.bronze_customers')
orders_items_df.write.format("delta").mode("overwrite").saveAsTable('ecommerce_pipeline.bronze.bronze_orders_items')
orders_payments_df.write.format("delta").mode("overwrite").saveAsTable('ecommerce_pipeline.bronze.bronze_orders_payments')
products_df.write.format("delta").mode("overwrite").saveAsTable('ecommerce_pipeline.bronze.bronze_products')

### Step 5: Read Bronze Tables

In [0]:
orders_df = spark.table("ecommerce_pipeline.bronze.bronze_orders")
customers_df = spark.table("ecommerce_pipeline.bronze.bronze_customers")
order_items_df = spark.table("ecommerce_pipeline.bronze.bronze_orders_items")
payments_df = spark.table("ecommerce_pipeline.bronze.bronze_orders_payments")
products_df = spark.table("ecommerce_pipeline.bronze.bronze_products")